<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Model_Load_Reproduce_ALIGNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%time
!pip install -q dgl==1.0.1+cu117 -f https://data.dgl.ai/wheels/cu117/repo.html
!pip install -q alignn

In [ ]:
import os
!pwd
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('alignn'):
  !git clone https://github.com/usnistgov/alignn.git

os.chdir('alignn')
# Install using setup.py in case pip didn't work
#!python setup.py develop

#!pip install dgl-cu111 # Colab has cuda 11.1

In [ ]:
import time
t1=time.time()
!train_folder.py --root_dir "alignn/examples/sample_data" --epochs 3 --batch_size 2 --config "alignn/examples/sample_data/config_example.json" --output_dir=temp
t2=time.time()
print ('Time in s',t2-t1)

In [ ]:
!ls -altr temp

In [ ]:
# from jarvis.core.atoms import Atoms
# from alignn.graphs import Graph
# cutoff = 8.0
# model=model.to(device)
# max_neighbors = 12
# atoms = Atoms.from_poscar('alignn/examples/sample_data/POSCAR-JVASP-10.vasp')
# g, lg = Graph.atom_dgl_multigraph(
#     atoms, cutoff=float(cutoff), max_neighbors=max_neighbors,
# )
# out_data = (
#     model([g.to(device), lg.to(device)])
#     .detach()
#     .cpu()
#     .numpy()
#     .flatten()
#     .tolist()
# )
# print ('output', out_data[0])

# Check after model reload we can reproduce predictions

In [ ]:
from alignn.models.alignn import ALIGNN, ALIGNNConfig
from jarvis.db.jsonutils import loadjson
from alignn.graphs import Graph
from jarvis.core.atoms import Atoms
import pandas as pd
from jarvis.db.figshare import data
import torch
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
outdir = 'temp'
rootdir = "alignn/examples/sample_data"
chkpt=os.path.join(outdir,"checkpoint_3.pt")
csv_path=os.path.join(outdir,"prediction_results_test_set.csv")
config_path=os.path.join(outdir,"config.json")
res_df = pd.read_csv(csv_path)
config=loadjson(config_path)
model = ALIGNN(ALIGNNConfig(**config["model"]))
model.load_state_dict(torch.load(chkpt, map_location=device)["model"])
model = model.to(device)
# from alignn.models.alignn import ALIGNN, ALIGNNConfig
# import torch
# output_features =  1
# filename = 'temp/checkpoint_3.pt'
# device = "cpu"
# if torch.cuda.is_available():
#     device = torch.device("cuda")
# model = ALIGNN(ALIGNNConfig(name="alignn", output_features=output_features))
# model.load_state_dict(torch.load(filename, map_location=device)["model"])
model.eval()

x=[]
y=[]
z=[]
aa=[]
for i,ii in (res_df.iterrows()):
  id=ii['id']
  pos_path=os.path.join(rootdir,id)
  atoms=Atoms.from_poscar(pos_path)
  aa.append(aa)
  g,lg=Graph.atom_dgl_multigraph(atoms,use_canonize=True,cutoff=8.0,neighbor_strategy="k-nearest",max_neighbors=12)
  g=g.to(device)
  lg=lg.to(device)
  model=model.to(device)
  pred=model([g,lg]).detach().cpu().numpy().tolist()

  print(id,pred,ii['prediction'],ii['target'])
  x.append(ii['target'])
  y.append(ii['prediction'])
  z.append(pred)
mae_old = mean_absolute_error(x,y)
mae_new = mean_absolute_error(x,z)
info={}

info['len']=len(x)
info['mae_old']=mae_old
info['mae_new']=mae_new
print(info)

In [ ]:
!train_folder_ff.py --root_dir "alignn/examples/sample_data_ff" --config "alignn/examples/sample_data_ff/config_example_atomwise.json" --output_dir=tempff

In [ ]:
!ls -altr tempff

In [ ]:
from alignn.graphs import Graph
from alignn.models.alignn_atomwise import ALIGNNAtomWise, ALIGNNAtomWiseConfig
config = loadjson("tempff/config.json")
path="tempff"
model_filename='best_model.pt'
model = ALIGNNAtomWise(ALIGNNAtomWiseConfig(**config["model"]))
model.state_dict()
model.load_state_dict(
    torch.load(
        os.path.join(path, model_filename), map_location=device
    )
)
model.to(device)
model.eval()

dat = loadjson("alignn/examples/sample_data_ff/id_prop.json")
ids_train_val_test = loadjson("tempff/ids_train_val_test.json")

In [ ]:
ids_train_val_test['id_test']

In [ ]:
test_res= loadjson('tempff/Val_results.json')
#id_prop = loadjson()

In [ ]:
test_res[0].keys()

In [ ]:
for ii,i in enumerate(dat[-2:]):
  atoms=Atoms.from_dict(i['atoms'])
  g,lg=Graph.atom_dgl_multigraph(atoms,use_canonize=True,cutoff=8.0,neighbor_strategy="k-nearest",max_neighbors=12)
  res=model([g,lg])
  print(i['total_energy'],test_res[ii]['pred_out'],res['out'])

In [ ]:
config

In [ ]:
test_res[0]